# Lesson 04 - Tiparul de proiectare Utilizarea Unealtelor

În această lecție veți învăța tiparul de proiectare **Utilizarea Unealtelor** pentru agenții AI folosind Microsoft Agent Framework (Python). Acoperim:

- Definirea uneltelor funcționale cu decoratorul `@tool` și parametri tipizați
- Oferirea schemelor pentru unelte astfel încât modelul să știe ce face fiecare unealtă
- Controlul execuției uneltelor cu `approval_mode`
- Returnarea unui **output structurat** prin modele Pydantic și `response_format`

Scenariul este un **agent de rezervare călătorii** care poate căuta destinații, verifica disponibilitatea și recupera informații despre zboruri.


## Configurare


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definirea uneltelor cu decoratorul @tool

Decoratorul `@tool` transformă o funcție Python simplă într-o unealtă pe care un agent o poate apela.  
Aspecte cheie:

- **docstring-ul** devine descrierea uneltei pe care modelul o vede.  
- **anotările de tip** (inclusiv `Annotated` cu descrieri) definesc schema uneltei.  
- `approval_mode` controlează dacă utilizatorul trebuie să aprobe fiecare apel înainte ca acesta să fie executat.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Crearea unui Agent cu Mai Multe Unelte

Transmită toate cele trei unelte clientului astfel încât modelul să poată invoca oricare dintre ele de care are nevoie pentru a răspunde la întrebarea utilizatorului.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Output Structurat cu Unelte

Prin setarea `response_format` la un model Pydantic, agentul este forțat să returneze un obiect JSON bine tipizat în loc de text liber. Acest lucru este util atunci când codul ulterior trebuie să consume rezultatul programatic.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tipare de Aprobare a Unei Unelte

Parametrul `approval_mode` din `@tool` controlează dacă apelurile uneltei necesită aprobarea unui om înainte de a fi executate:

| Mod | Comportament |
|---|---|
| `"never_require"` | Unealta rulează automat — nu este necesară confirmarea utilizatorului. |
| `"always_require"` | Fiecare apel trebuie aprobat de utilizator înainte de a fi executat. |

Folosește `"always_require"` pentru unelte care au efecte secundare (ex. rezervarea unui bilet de avion, încasarea unei sume pe cardul de credit) astfel încât un om să rămână în buclă.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Rezumat

În această lecție ai învățat cum să:

1. **Definiți unelte** folosind decoratorul `@tool` cu parametri tipizați și docstring-uri care servesc drept schema uneltei.
2. **Compoziți mai multe unelte** astfel încât agentul să le poată apela în succesiune pentru a răspunde la interogări complexe.
3. **Returnați un rezultat structurat** prin transmiterea unui model Pydantic ca `response_format`.
4. **Controlați aprobarea uneltelor** cu `approval_mode` pentru a menține un om în circuit în cazul operațiunilor sensibile.

Aceste modele formează baza pentru construirea unor agenți fiabili, gata de producție, care pot interacționa în siguranță cu sisteme externe.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus utilizând serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea umană profesională. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite rezultate din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
